In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

df = pd.read_csv(r'C:\DATA ANALYSIS AND DATA SCIENCE PROJECTS\Accident Severity Prediction\data\clean data\final_dataset.csv') 
df.head()


,Unnamed: 0,State_Code,State_Name,Crash_ID,Count_Non_Motorists,Count_Vehicles_Total,Count_Vehicle_Forms,Count_PVH_Involved,Count_Occupants_MV,City_Name,...,Total_Exceeding_Limit_Vehicles,Total_Too_Fast_Vehicles,Max_Vehicle_Type_Severity_5,Max_Vehicle_Damage_Rank,Max_Harmful_Event_Severity,Crash_Road_Surface_Condition,Crash_Horizontal_Alignment,Crash_Vertical_Profile,Crash_Impairment_Category,Crash_Vehicle_Defect
0,0,1,Alabama,10001,0,1,1,0,1,NOT APPLICABLE,...,0,0,3,4,2,Dry,Straight,Grade/Hillcrest,Medium,No
1,1,1,Alabama,10002,0,1,1,0,1,NOT APPLICABLE,...,0,0,2,4,3,Dry,Straight,Level,Lowest,No
2,2,1,Alabama,10003,0,1,1,0,1,NOT APPLICABLE,...,1,0,2,4,3,Dry,Curve,Grade/Hillcrest,Lowest,No
3,3,1,Alabama,10004,0,1,1,0,3,NOT APPLICABLE,...,3,0,3,4,3,Dry,Curve,Grade/Hillcrest,Lowest,No
4,4,1,Alabama,10005,0,1,1,0,4,NOT APPLICABLE,...,4,0,2,4,3,Wet/Standing Water,Straight,Level,Lowest,No


In [26]:
def map_fatality_level(x):
    if x == 1:
        return 'Low Fatality'
    else:
        return 'High Fatality'

df['Fatality_Level'] = df['Fatalities_Count'].apply(map_fatality_level)
df['Fatality_Level_Binary'] = df['Fatality_Level'].map({'Low Fatality': 0, 'High Fatality': 1})
print(df['Fatality_Level_Binary'].value_counts())

Fatality_Level_Binary
0    35019
1     2635
Name: count, dtype: int64


In [27]:
df["Restraint_Used"] = df["Killed_Restraint_List"].astype(str).str.contains(
    "Lapbelt|Shoulder_Lapbelt", case=False, na=False).astype(int)

features = df.copy()
drop_cols = [
    "Crash_ID", "State_Name", "State_Code", "City_Name",
    "Notif_Hour", "Notif_Minute", 
    "Arrival_Hour", "Arrival_Minute",
    "Hospital_Hour", "Hospital_Minute",
    "Fatalities_Count", "Total_Killed", "Total_Alive",
    "Fatal_Seats_List", "Killed_Restraint_List",
    "Latitude", "Longitude"
]
features = features.drop(columns=drop_cols, errors='ignore')

numeric_features = [
    'Count_Vehicles_Total', 'Count_Occupants_MV', 'Count_Non_Motorists',
    'Crash_Hour', 'Crash_Minute',
    'Max_Driver_Age', 'Min_Driver_Age',
    'Total_Exceeding_Limit_Vehicles', 'Total_Too_Fast_Vehicles',
    'Driver_Max_Prev_Accident_Count', 'Driver_Max_Prev_Speeding_Count'
]

categorical_features = [
    'Clean_Light_Condition', 'Clean_Weather_Condition',
    'Area_Rural_Urban', 'Road_Functional_Class',
    'Intersection_Type', 'Driver_License_Category',
    'Light_Condition', 'Weather_Condition',
    'Crash_Road_Surface_Condition', 'Crash_Horizontal_Alignment',
    'Crash_Vertical_Profile', 'Crash_Impairment_Category',
    'Crash_Vehicle_Defect', 'On_NHS_Road',
    'Any_Driver_Drinking', 'Driver_Sex_List'
]

In [28]:
target_column = 'Fatality_Level_Binary'
X = features  
y = df[target_column]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [29]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numeric_features])
X_test_num = scaler.transform(X_test[numeric_features])
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])
X_train_processed = np.hstack((X_train_num, X_train_cat))
X_test_processed = np.hstack((X_test_num, X_test_cat))


In [30]:
lr = LogisticRegression(max_iter=500, class_weight='balanced')
lr.fit(X_train_processed, y_train)

y_pred_lr = lr.predict(X_test_processed)
y_proba_lr = lr.predict_proba(X_test_processed)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nROC AUC:", roc_auc_score(y_test, y_proba_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

Accuracy: 0.7629796839729119

ROC AUC: 0.8156897335276398

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.77      0.86      6982
           1       0.19      0.70      0.30       549

    accuracy                           0.76      7531
   macro avg       0.58      0.74      0.58      7531
weighted avg       0.91      0.76      0.82      7531


Confusion Matrix:
 [[5359 1623]
 [ 162  387]]
